In [28]:
%load_ext autoreload
%autoreload 2

# Import the necessary libraries 
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

from src_prognostics.utils import load_data, format_data
from src_prognostics.hsmm import CustomHSMM, predict_hsmm_pdf_staked, predict_hsmm_bounds
from src_prognostics.metrics import mae, picp, pinaw

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Assignment

You recive a dataset that contains condition monitoring (CM) data from a structure. The dataset contrains several run to failure experiments that will allow you to train a hidden semi-Markov model (HSMM) for remaining useful life (RUL) prognostics. In addition to the CM data, dataset indicates the operational conditions in which the expreminets were perfomed.

This workshop considers the following tasks:
1. Visulize the provided data, explore its contents and gather some insights
2. Use the provided codes to train a baseline HSMM. Reflect on the achieved accuracy and uncertainty representation.
3. Reflect on how you incorporate uncertainty management in the HSMM.
4. Implemet your uncertainty management strategy and analyze its effectivity
5. Compute all prognostic distributions for all training, validation and testing datasets. These results will be latter used for deicission-making.

The following cells contain some code examples that you could use. Also, some functionalies are already implemeted.in the source code. Feel free to use them. 
In case you have questions, or need assistence, don't hessitate to ask for help 😁


## Task 1: Analyze the data

In [2]:
# To load the dataset
df_train = load_data()
df_train.head()

,hi,load,exp_id
0,-0.000000,0.0,0.0
1,0.001233,0.0,0.0
2,0.002353,0.0,0.0
3,0.002721,0.0,0.0
4,0.003831,0.0,0.0


In [3]:
# Plot some columns of the dataset here

In [4]:
# Plot some columns of the dataset here

In [5]:
# Plot some columns of the dataset here

What insigts did you get from the data?

Write your answer here:



...

## Task 2: Train an HSMM

Here are some implemented codes that can train an HSMM

In [6]:
# Transform the data to the HIMAP format
data_train, max_len = format_data(df_train)

In [7]:
# Instance the model with some hiper-parameters
n_states = 4
model = CustomHSMM(n_states=n_states, # You can change this
                   n_durations = (max_len//n_states)*2, # You can change this
                   f_value=-0.34, # Please, dont change this.
                   obs_state_len=len(data_train.keys()), # Please, dont change this.
                   n_iter=3, # You can change this
                   name = 'hsmm_exmple' # You can change this
                   )

Folder already exists: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results
Folder already exists: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\dictionaries
Folder already exists: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\figures
Folder already exists: c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\models


In [8]:
# Fit the model parameters
model.fit(data_train)

Iters 1/3: 100%|██████████| 20/20 [00:04<00:00,  4.14it/s]



FIT hsmm_exmple: re-estimation complete for loop 1 with score: -2871.514264571337.


Iters 2/3: 100%|██████████| 20/20 [00:04<00:00,  4.05it/s]



FIT hsmm_exmple: re-estimation complete for loop 2 with score: -118.78059879975862.


Iters 3/3: 100%|██████████| 20/20 [00:04<00:00,  4.05it/s]


FIT hsmm_exmple: re-estimation complete for loop 3 with score: -93.90591065648691.
[0, 1, 2, 3]


In [9]:
# Save the model for later
model.save_model()

Model saved to c:\Users\bbritoschiele\Documents\PHD_files\2026\summer_school_prog_dm\himap_results\models\hsmm_exmple.pkl.


In [ ]:
# In case you need to load a trained model
model = CustomHSMM(name = 'hsmm_exmple')
model.load_model('hsmm_exmple')

In [36]:
alpha = 0.005 # Confidence level used to generate the intervals
print('Mae\tPINAW\tPICP')
for cmd in data_train.values():
    
    actual_RUL = len(cmd)-np.arange(len(cmd))
    
    expected, lb, ub = predict_hsmm_bounds(cmd, model,alpha = alpha ) # Function that returns the expected rul and lower and upper values for a confidence intervals
    
    print(mae(expected,actual_RUL), '\t', pinaw(lb,ub),'\t', picp(lb,ub, actual_RUL))
    
    # # Uncomment these lines to get the RUL plot
    # x_axis = np.arange(len(cmd))
    # plt.figure()
    # plt.plot(x_axis,expected, label = 'Prognostic', color = 'b')
    # plt.fill_between(x_axis, lb, ub, alpha = 0.2, color = 'b')
    # plt.plot(x_axis,actual_RUL, color = 'r', label = 'Actual RUL')
    # plt.ylabel('RUL')
    # plt.xlabel('Timestep')
    # plt.legend()
    # plt.show()
    # break

    
    

Mae	PINAW	PICP
32.850349861366816 	 124.42245989304813 	 0.893048128342246
21.710136721960662 	 118.87974683544304 	 0.8734177215189873
29.073779077544202 	 119.23563218390805 	 0.8850574712643678
23.65384946713138 	 123.60714285714286 	 0.8809523809523809
35.493967540592486 	 117.97267759562841 	 0.8907103825136612
26.101272713860233 	 121.62941176470588 	 0.8823529411764706
40.37701690525757 	 117.94270833333333 	 0.8958333333333334
19.488087970007477 	 126.81212121212121 	 0.8787878787878788
21.218573958161258 	 123.77439024390245 	 0.8780487804878049
49.770910342438995 	 112.42156862745098 	 0.8970588235294118
7.096621228220127 	 126.096 	 0.84
11.402459003220782 	 123.91379310344827 	 0.8275862068965517
7.660213085082538 	 124.48360655737704 	 0.8360655737704918
4.490674918709495 	 127.71851851851852 	 0.8518518518518519
6.807686677100902 	 121.02479338842976 	 0.8347107438016529
6.752453665177863 	 120.60740740740741 	 0.8518518518518519
3.7041461664173476 	 124.04615384615384 	 

How is the uncertainty of the model?


## Task 5: Compute your prognostics for the training, validation and testing datasets